In [154]:
# ============================================================
# STEP 1 — Install the packages needed for the project
# ============================================================

!pip -q install \
    requests==2.32.4 \
    weaviate-client \
    langchain-core \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-weaviate \
    sentence-transformers \
    pypdf \
    huggingface_hub

In [155]:
# ============================================================
# STEP 2 — Import all required libraries
# ============================================================
'''from google.colab import userdata


# Weaviate client
import weaviate
from weaviate.classes.init import Auth

# LangChain components
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFaceEndpoint
from langchain_weaviate import WeaviateVectorStore

# Chain building tools
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser'''

'from google.colab import userdata\n\n\n# Weaviate client\nimport weaviate\nfrom weaviate.classes.init import Auth\n\n# LangChain components\nfrom langchain_community.document_loaders import PyPDFLoader\nfrom langchain_text_splitters import RecursiveCharacterTextSplitter\nfrom langchain_huggingface import HuggingFaceEmbeddings, HuggingFaceEndpoint\nfrom langchain_weaviate import WeaviateVectorStore\n\n# Chain building tools\nfrom langchain_core.prompts import ChatPromptTemplate\nfrom langchain_core.runnables import RunnablePassthrough\nfrom langchain_core.output_parsers import StrOutputParser'

In [156]:
# STEP 3 — Store credentials in environment variables
import os
import getpass

os.environ["WEAVIATE_URL"] = userdata.get('WEAVIATE_URL')
os.environ["WEAVIATE_API_KEY"] = userdata.get('WEAVIATE_API_KEY')
os.environ["HUGGINGFACE_API_TOKEN"] = userdata.get('HUGGINGFACE_TOKEN')

# reading them
weaviate_url = os.environ["WEAVIATE_URL"]
weaviate_api_key = os.environ["WEAVIATE_API_KEY"]
huggingface_api_key = os.environ["HUGGINGFACE_API_TOKEN"]



In [157]:
# ============================================================
# STEP 4 — Connect to Weaviate Cloud
# ============================================================

## the code below does the following:

# reads the credentials from environment variables
# connects your notebook to your Weaviate Cloud cluster
# checks if the connection is working with   `client.is_ready()`
import weaviate
from weaviate.classes.init import Auth

client = weaviate.connect_to_weaviate_cloud(
    cluster_url = weaviate_url,
    auth_credentials=Auth.api_key(weaviate_api_key),
)

print(client.is_ready())

True


In [158]:
# ============================================================
# STEP 5 — Load the PDF file
# ============================================================

## What this step does:

# opens your PDF file
# converts each page into a LangChain Document
# keeps metadata such as page number and source path, which is useful in RAG workflows

from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "/content/Retrieval-Augmented_Generation_RAG.pdf"
loader = PyPDFLoader(PDF_PATH)
pages = loader.load_and_split()



In [159]:
# ============================================================
# STEP 6 — Split the PDF text into smaller chunks
# ============================================================

## What this step does:

# breaks long pages into smaller text chunks
# chunking helps retrieval work better because vector databases search smaller, more focused pieces of text
# overlap keeps some shared context between neighboring chunks


from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap  = 150
)

docs = text_splitter.split_documents(pages)


In [160]:
print (f'number of Chunks after splitting: {len(docs)}')

number of Chunks after splitting: 82


In [161]:
print(docs[1].page_content)

This has led to a need for new approaches that can better
address the information needs of organizations.
Conversational agents (CAs) powered by Artiﬁcial
Intelligence (AI) and transformer-based large language
models (LLMs) in particular (Vaswani et al. 2017) have
revolutionized the way information can be accessed today.
When compared to traditional enterprise systems, CAs
offer two key beneﬁts: Firstly, they enable users to pose
questions in a natural and intuitive manner using natural
language, receiving responses that are similarly conversa-
tional. Secondly, they are increasingly capable of tackling
complex search tasks, facilitating problem-solving and
decision-making in various domains (White 2024). For
instance, individuals can use CAs to access recipe infor-
mation for cooking (Jaber et al. 2024) and to obtain
assistance with complex chemistry-related tasks (Bran
et al. 2024) and geometric problems (Trinh et al. 2024).


In [162]:
# ============================================================
# STEP 7 — Create embeddings using Hugging Face
# ============================================================

## What this step does:

# loads a sentence-transformer model from Hugging Face
# converts each text chunk into a vector embedding
# external embeddings, not Weaviate-managed vectorization

embedding_model_name = "sentence-transformers/all-mpnet-base-v2"

from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name = embedding_model_name
)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [163]:
# ============================================================
# STEP 8 — Create a Weaviate vector store and index the chunks
# ============================================================

## What this step does:

# creates a Weaviate-backed vector store through LangChain
# embeds each chunk using Hugging Face
# stores the chunks and their vectors in Weaviate
# langchain-weaviate is the current LangChain package for this integration

from langchain_weaviate import WeaviateVectorStore

INDEX_NAME = "RagPdfDemo"
TEXT_KEY = "text"

vector_db = WeaviateVectorStore.from_documents(
    documents=docs,
    embedding=embeddings,
    client=client,
    index_name=INDEX_NAME,
    text_key=TEXT_KEY,
)

print(f"Documents indexed in Weaviate collection: {INDEX_NAME}")

Documents indexed in Weaviate collection: RagPdfDemo


In [165]:
# ============================================================
# STEP 9 — Test similarity search
# ============================================================
query = "what is retrieval augmented generation?"

results = vector_db.similarity_search(query, k=3)

for i, doc in enumerate(results, start=1):
    print(f"\n========== Result {i} ==========\n")
    print(doc.metadata)
    print(doc.page_content)
    #print("\n")


========== Result 1 ==========

{'page_label': '8', 'creationdate': datetime.datetime(2025, 9, 18, 3, 3, 54, tzinfo=datetime.timezone(datetime.timedelta(seconds=7200))), 'source': '/content/Retrieval-Augmented_Generation_RAG.pdf', 'moddate': datetime.datetime(2025, 9, 18, 3, 3, 54, tzinfo=datetime.timezone(datetime.timedelta(seconds=7200))), 'creator': 'PyPDF', 'total_pages': 12.0, 'producer': 'iText® 5.5.13.3 ©2000-2022 iText Group NV (SPRINGER SBM; licensed version)', 'page': 7.0}
ﬁrst?) and how well the retrieval process performs.
Fig. 3 Research questions related to RAG
123
558 M. Klesel, H. F. Wittmann: Retrieval-Augmented Generation (RAG), Bus Inf Syst Eng 67(4):551–561 (2025)
Content courtesy of Springer Nature, terms of use apply. Rights reserved.

========== Result 2 ==========

{'page_label': '8', 'creationdate': datetime.datetime(2025, 9, 18, 3, 3, 54, tzinfo=datetime.timezone(datetime.timedelta(seconds=7200))), 'moddate': datetime.datetime(2025, 9, 18, 3, 3, 54, tzinfo=dat

In [166]:
# ============================================================
# STEP 10 — Create a retriever from the vector store
# ============================================================

## What this step does:

# turns the vector database into a retriever object
# this retriever will fetch relevant chunks whenever the user asks a question
# k=4 means retrieve the top 4 most relevant chunks

retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

retriever

VectorStoreRetriever(tags=['WeaviateVectorStore', 'HuggingFaceEmbeddings'], vectorstore=<langchain_weaviate.vectorstores.WeaviateVectorStore object at 0x7be61460ab40>, search_kwargs={'k': 4})

In [167]:
# ============================================================
# STEP 11 — Create the prompt template
# ============================================================

## What this step does:

# defines how the retrieved context and the question will be sent to the model
# tells the model not to invent answers if the information is missing

from langchain_core.prompts import ChatPromptTemplate

template = """You are an assistant for question-answering tasks.
Use the retrieved context below to answer the question.
If the answer is not in the context, say you don't know.
Keep the answer clear and concise.

Question:
{question}

Context:
{context}

Answer:
"""



prompt = ChatPromptTemplate.from_template(template)

print("Prompt template is ready.")

Prompt template is ready.


In [168]:
# ============================================================
# STEP 12 — Load the Mistral model via Hugging Face Endpoint
# ============================================================

## What this step does:

# connects to the Qwen model through Hugging Face
# this model will generate the final answer using the retrieved context

from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="conversational",
    temperature=0.3,
    max_new_tokens=256,
    huggingfacehub_api_token = huggingface_api_key,
)

# Chat wrapper
chat_model = ChatHuggingFace(llm=llm)


In [170]:
# testing the mode
print(chat_model.invoke("when was messi born.").content)

Lionel Messi was born on June 24, 1987.


In [171]:
# ============================================================
# STEP 13 — Build the RAG chain
# ============================================================

## What this step does:

# retrieves relevant chunks from Weaviate
# combines them into one context string
# sends the question + context to the prompt
# sends that prompt to Qwen
# converts the model output into plain text

from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(retrieved_docs):
    """Combine retrieved documents into one text block."""
    return "\n\n".join(doc.page_content for doc in retrieved_docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | chat_model
    | StrOutputParser()
)

In [172]:
# ============================================================
# STEP 14 — Ask your first question
# ============================================================
answer = rag_chain.invoke("What is RAG and how does it work?")
print(answer)

RAG stands for Retrieval-Augmented Generation. It works by using a large language model (LLM) and supplementing it with external data during the generation process. This external data is stored in a dedicated database and is distinct from the LLM's parametric memory. The LLM retrieves relevant information from this database to generate more accurate and contextually rich responses to queries. This approach allows organizations to incorporate their internal data and contextual knowledge, enhancing the LLM's ability to provide relevant information.


In [173]:
# ============================================================
# STEP 14 — Ask your first question
# ============================================================
answer = rag_chain.invoke("what are the applications of rag? why do we use rag?")
print(answer)

RAG (Retrieval-Augmented Generation) can be used to integrate additional data sources to provide real-time information retrieval. We use RAG to improve the timeliness and accuracy of results, especially in applications that require specific information. For example, it can be used in AI-empowered programming assistants and the ChatGPT Android app to retrieve information from the web in real time.
